# 03 · Formulación extensa (MILP)

En esta parte se plantea el problema completo como un único programa lineal entero mixto (MILP), incluyendo explícitamente los $K$ escenarios en el mismo modelo (sin descomposición).

Este notebook sirve como **referencia de validación** frente al algoritmo *Integer L-shaped* (`04_lshaped.ipynb`) sobre la muestra de $K=50$ escenarios generada en el notebook anterior (sección 5.1).  
Ambas implementaciones deben coincidir en el valor objetivo dentro de una tolerancia relativa de $10^{-4}$.

**Archivos generados:**
- `data/processed/extensive_form_K50_solution.json` → contiene la ruta, el valor objetivo, el tamaño del modelo y el tiempo de solución.

In [1]:
import time
import json
import numpy as np
import pandas as pd
import pulp
from pathlib import Path

PROC_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results/tables")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sites = pd.read_csv(PROC_DIR / "sites.csv")
arc_order = pd.read_csv(PROC_DIR / "arc_order.csv")
arc_costs = pd.read_csv(PROC_DIR / "arc_costs.csv")
scenarios = pd.read_parquet(PROC_DIR / "scenarios_K50.parquet")

K = len(scenarios)
print(f"K = {K} escenarios cargados")
print(f"Arcos: {len(arc_order)}")


K = 50 escenarios cargados
Arcos: 110


## 1. Conjuntos y parámetros

$V = \{0, 1, \dots, 10\}$, $N = V \setminus \{0\}$, $A = \{(i,j) \in V \times V : i \neq j\}$, $n = |N| = 10$.

Parámetros fijos (sección 3 del enunciado).

In [2]:
V = sites["i"].tolist()          # 0..10
N = [i for i in V if i != 0]      # 1..10
A = list(zip(arc_order["i"], arc_order["j"]))
n = len(N)

c_ij = {(row.i, row.j): row.c_ij for row in arc_costs.itertuples()}

H = 390.0        # horizonte operativo regular [min]
o_bar = 30.0      # tiempo adicional ordinario máximo [min]
c_OT = 1.50       # costo tiempo adicional ordinario [dólares/min]
c_EM = 6.00       # costo sobretiempo de emergencia [dólares/min]
alpha = {i: 1.0 for i in N}  # productividad en sitio

demand_cost = pd.DataFrame([
    {"i": 1,  "d_i": 30, "c_out_i": 2.60},
    {"i": 2,  "d_i": 30, "c_out_i": 2.40},
    {"i": 3,  "d_i": 35, "c_out_i": 2.80},
    {"i": 4,  "d_i": 30, "c_out_i": 2.30},
    {"i": 5,  "d_i": 25, "c_out_i": 2.10},
    {"i": 6,  "d_i": 25, "c_out_i": 2.00},
    {"i": 7,  "d_i": 35, "c_out_i": 2.70},
    {"i": 8,  "d_i": 40, "c_out_i": 3.00},
    {"i": 9,  "d_i": 35, "c_out_i": 2.90},
    {"i": 10, "d_i": 25, "c_out_i": 2.20},
]).set_index("i")

d_i = demand_cost["d_i"].to_dict()
c_out_i = demand_cost["c_out_i"].to_dict()

print(f"n = {n}, |A| = {len(A)}, sum(d_i) = {sum(d_i.values())} min")


n = 10, |A| = 110, sum(d_i) = 310 min


## 2. Vector de escenarios en formato utilizable

Convertimos `scenarios` (formato ancho, columnas `xi_i_j`) a un diccionario `xi[s][(i,j)]` para indexar fácilmente dentro del modelo.

In [3]:
xi = {}
for s in range(K):
    xi[s] = {
        (i, j): scenarios.loc[s, f"xi_{i}_{j}"]
        for (i, j) in A
    }

print(f"Ejemplo escenario 0, arco {A[0]}: {xi[0][A[0]]:.2f} min")


Ejemplo escenario 0, arco (0, 1): 3.18 min


## 3. Construcción del modelo MILP

**Primera etapa**
- $x_{ij} \in \{0,1\}$ para $(i,j) \in A$: uso del arco.
- $u_i$ continuas (variables de orden MTZ) para $i \in N$, con $1 \le u_i \le n$.
- Restricciones de grado (entra/sale una vez de cada nodo) + eliminación de subtours (MTZ).

**Segunda etapa** (para cada escenario $s = 1, \dots, K$)
- $u_i^{(s)} \in [0, d_i]$: minutos ejecutados directamente.
- $r_i^{(s)} \ge 0$: minutos tercerizados.
- $o^{(s)} \in [0, \bar o]$: tiempo adicional ordinario.
- $e^{(s)} \ge 0$: sobretiempo de emergencia.

**Notación:** usamos `w[i,s]` para los minutos directos de segunda etapa (evita chocar con la variable de orden MTZ `u[i]` de primera etapa, que en el enunciado también se llama $u_i$ pero corresponde a un objeto distinto).

In [4]:
prob = pulp.LpProblem("extensive_form_SAA", pulp.LpMinimize)

# --- Primera etapa ---
x = pulp.LpVariable.dicts("x", A, cat="Binary")
u_order = pulp.LpVariable.dicts("u_order", N, lowBound=1, upBound=n, cat="Continuous")

# Restricciones de grado: sale una vez de cada nodo, entra una vez a cada nodo
for k in V:
    prob += pulp.lpSum(x[(k, j)] for (i, j) in A if i == k) == 1, f"out_degree_{k}"
    prob += pulp.lpSum(x[(i, k)] for (i, j) in A if j == k) == 1, f"in_degree_{k}"

# Eliminación de subtours (MTZ), solo entre nodos de N (no involucra al depósito 0)
for (i, j) in A:
    if i in N and j in N:
        prob += u_order[i] - u_order[j] + n * x[(i, j)] <= n - 1, f"mtz_{i}_{j}"

# --- Segunda etapa: variables por escenario ---
w = pulp.LpVariable.dicts("w", [(i, s) for i in N for s in range(K)], lowBound=0)       # minutos directos
r = pulp.LpVariable.dicts("r", [(i, s) for i in N for s in range(K)], lowBound=0)       # minutos tercerizados
o = pulp.LpVariable.dicts("o", range(K), lowBound=0, upBound=o_bar)                     # tiempo adicional ordinario
e = pulp.LpVariable.dicts("e", range(K), lowBound=0)                                    # sobretiempo de emergencia

for i in N:
    for s in range(K):
        prob += w[(i, s)] <= d_i[i], f"w_ub_{i}_{s}"
        prob += w[(i, s)] + r[(i, s)] == d_i[i], f"demanda_{i}_{s}"

for s in range(K):
    prob += (
        pulp.lpSum(xi[s][(i, j)] * x[(i, j)] for (i, j) in A)
        + pulp.lpSum(alpha[i] * w[(i, s)] for i in N)
        <= H + o[s] + e[s]
    ), f"tiempo_{s}"

# --- Función objetivo ---
first_stage_cost = pulp.lpSum(c_ij[(i, j)] * x[(i, j)] for (i, j) in A)
recourse_cost = (1 / K) * pulp.lpSum(
    pulp.lpSum(c_out_i[i] * r[(i, s)] for i in N) + c_OT * o[s] + c_EM * e[s]
    for s in range(K)
)
prob += first_stage_cost + recourse_cost

n_vars = prob.numVariables()
n_constrs = prob.numConstraints()
print(f"Variables: {n_vars}")
print(f"Restricciones: {n_constrs}")


Variables: 1220
Restricciones: 1162


## 4. Resolver el modelo

Se usa el solver **CBC** (incluido con PuLP). Se mide el tiempo de solución para comparar después con el Integer L-shaped y con la aproximación del perfil promedio (sección 5.3, punto 5).

In [6]:
# --- Selección automática del solver CBC ---
# El binario de CBC que trae PuLP empaquetado no siempre es compatible con todas las
# arquitecturas de procesador (por ejemplo, Apple Silicon) ni con todos los sistemas
# operativos. Por portabilidad y reproducibilidad, se detecta automáticamente un CBC
# instalado en el sistema (vía PATH o rutas de instalación típicas en Mac/Linux/Windows)
# y se usa en su lugar; si no se encuentra ninguno, se recurre al binario empaquetado
# con PuLP como respaldo.
import shutil
import sys


def get_cbc_solver(msg: bool = False) -> pulp.LpSolver:
    """Detecta y retorna el mejor solver CBC disponible en este sistema."""
    candidate_paths = [
        shutil.which("cbc"),
        shutil.which("cbc.exe"),
        "/opt/homebrew/bin/cbc",                                  # Homebrew, Apple Silicon
        "/usr/local/bin/cbc",                                     # Homebrew, Mac Intel / Linux
        "/usr/bin/cbc",                                           # gestores de paquetes de Linux
        str(Path(sys.prefix) / "Library" / "bin" / "cbc.exe"),    # conda, Windows
        str(Path(sys.prefix) / "bin" / "cbc"),                    # conda, Mac/Linux
    ]
    cbc_path = next((p for p in candidate_paths if p and Path(p).exists()), None)
    if cbc_path:
        return pulp.COIN_CMD(msg=msg, path=cbc_path)
    return pulp.PULP_CBC_CMD(msg=msg)


solver = get_cbc_solver(msg=True)

# --- Resolver el modelo ---
t0 = time.time()
prob.solve(solver)
solve_time = time.time() - t0

status = pulp.LpStatus[prob.status]
objective = pulp.value(prob.objective)

print(f"Estado: {status}")
print(f"Valor objetivo: {objective:.4f}")
print(f"Tiempo de solución: {solve_time:.2f} s")

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - /opt/homebrew/bin/cbc /var/folders/40/npdd49x91951z39cxmk17hcw0000gn/T/217f93a27e7b4686aaad94fb91d75d91-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/40/npdd49x91951z39cxmk17hcw0000gn/T/217f93a27e7b4686aaad94fb91d75d91-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 1167 COLUMNS
At line 10188 RHS
At line 11351 BOUNDS
At line 11532 ENDATA
Problem MODEL has 1162 rows, 1220 columns and 8090 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 29.1361 - 0.00 seconds
Cgl0003I 0 fixed, 0 tightened bounds, 90 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 90 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 90 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 90 strengthened rows, 0 substitutions
Cg

## 5. Extraer la ruta óptima

Reconstruimos la secuencia de visita a partir de los arcos con $x_{ij} = 1$, partiendo del depósito (nodo 0).

In [6]:
selected_arcs = [(i, j) for (i, j) in A if pulp.value(x[(i, j)]) > 0.5]
print(f"Arcos seleccionados: {len(selected_arcs)}")

# Reconstrucción de la ruta como secuencia de nodos
next_node = {i: j for (i, j) in selected_arcs}
route = [0]
current = 0
for _ in range(len(V) - 1):
    current = next_node[current]
    route.append(current)
route.append(0)  # regreso al depósito

# Control de calidad: la ruta debe visitar cada nodo exactamente una vez (más el regreso)
assert len(set(route[:-1])) == len(V), "La ruta no visita todos los nodos exactamente una vez."
assert route[0] == 0 and route[-1] == 0, "La ruta no parte ni regresa al depósito."

route_names = sites.set_index("i")["name"]
print("Ruta óptima x*:")
print(" -> ".join(route_names[node] for node in route))


Arcos seleccionados: 11
Ruta óptima x*:
Javits Center (Depósito) -> Madison Square Garden -> Times Square -> Rockefeller Center -> Grand Central Terminal -> New York Public Library -> Union Square -> Washington Square Park -> South Street Seaport (Pier 17) -> New York Stock Exchange -> One World Trade Center -> Javits Center (Depósito)


## 6. Diagnóstico del recurso (frecuencias de uso)

Para cuántos de los $K=50$ escenarios se activa tiempo adicional, tercerización y sobretiempo de emergencia.

In [7]:
freq_o = sum(1 for s in range(K) if pulp.value(o[s]) > 1e-6) / K
freq_e = sum(1 for s in range(K) if pulp.value(e[s]) > 1e-6) / K
freq_r = sum(
    1 for s in range(K)
    if any(pulp.value(r[(i, s)]) > 1e-6 for i in N)
) / K

print(f"Frecuencia de uso de tiempo adicional ordinario (o>0): {freq_o:.1%}")
print(f"Frecuencia de uso de sobretiempo de emergencia (e>0):  {freq_e:.1%}")
print(f"Frecuencia de uso de tercerización (algún r_i>0):      {freq_r:.1%}")


Frecuencia de uso de tiempo adicional ordinario (o>0): 92.0%
Frecuencia de uso de sobretiempo de emergencia (e>0):  0.0%
Frecuencia de uso de tercerización (algún r_i>0):      22.0%


## 7. Guardar resultados

Estos resultados se comparan directamente en `04_lshaped.ipynb` (mismo K=50, misma tolerancia relativa $10^{-4}$).

In [8]:
result = {
    "K": K,
    "status": status,
    "objective": objective,
    "solve_time_sec": solve_time,
    "n_variables": n_vars,
    "n_constraints": n_constrs,
    "route": route,
    "freq_overtime": freq_o,
    "freq_emergency": freq_e,
    "freq_outsourcing": freq_r,
}

with open(RESULTS_DIR / "extensive_form_K50_solution.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Guardado en {RESULTS_DIR / 'extensive_form_K50_solution.json'}")
result


Guardado en ../results/tables/extensive_form_K50_solution.json


{'K': 50,
 'status': 'Optimal',
 'objective': 52.88447674799998,
 'solve_time_sec': 0.5438430309295654,
 'n_variables': 1220,
 'n_constraints': 1162,
 'route': [0, 7, 1, 2, 3, 4, 5, 6, 10, 9, 8, 0],
 'freq_overtime': 0.92,
 'freq_emergency': 0.0,
 'freq_outsourcing': 0.22}